# Clase 7 — Pipeline ML completo

En la clase anterior entrenamos un modelo en pasos sueltos. El problema con eso es que si cambiamos los datos o queremos repetir el experimento en otro equipo, hay que recordar el orden exacto y los parámetros de cada paso. El objeto **`Pipeline` de scikit-learn** resuelve esto: encadena preprocesamiento y modelo en un solo objeto reproducible.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Por qué usar un pipeline |
| 2 | El dataset: Titanic simplificado |
| 3 | Preprocesamiento: numérico y categórico |
| 4 | Construir el pipeline |
| 5 | Entrenar, evaluar y reutilizar |
| 6 | Guardar y cargar el pipeline |
| 7 | Actividad: pipeline propio con un dataset nuevo |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

print("Librerías listas.")

---
## 1. Por qué usar un pipeline

Cuando aplicás transformaciones antes de entrenar (escalar, imputar, codificar), hay una trampa frecuente: **data leakage** — filtrar información del test set al proceso de entrenamiento.

| Flujo incorrecto | Flujo correcto con Pipeline |
|---|---|
| Escalás **todo** el dataset, luego dividís | Dividís primero, luego el pipeline escala solo con datos de train |
| El scaler "vio" los datos de test | El scaler solo aprende la media/std del train |
| La evaluación es optimista y engañosa | La evaluación refleja el rendimiento real |

> 💡 `Pipeline.fit(X_train)` ajusta todos los pasos con train. `Pipeline.transform(X_test)` aplica las transformaciones ya ajustadas — sin re-aprender. Eso es reproducibilidad real.

---
## 2. El dataset: Titanic simplificado

Vamos a usar el dataset del Titanic, disponible en seaborn. La tarea es predecir si un pasajero sobrevivió. Tiene features numéricas (`age`, `fare`) y categóricas (`sex`, `class`), lo que lo hace ideal para practicar pipelines con preprocesamiento mixto.

In [ ]:
# ─── Cargar y limpiar el dataset ──────────────────────────────────────────────
import seaborn as sns

df = sns.load_dataset("titanic")

# Seleccionamos solo las columnas que nos interesan para este ejercicio
features = ["pclass", "sex", "age", "fare", "embark_town"]
target   = "survived"

df = df[features + [target]].copy()

print(f"Shape: {df.shape}")
print()
print("Primeras filas:")
df.head()

In [ ]:
# ─── Explorar valores nulos ───────────────────────────────────────────────────
print("Valores nulos por columna:")
print(df.isnull().sum())
print()
print("Balance de clases:")
print(df[target].value_counts())
print(f"  ({df[target].mean():.1%} de sobrevivientes)")

In [ ]:
# ─── Separar features y etiqueta ─────────────────────────────────────────────
# Eliminamos filas sin etiqueta (si las hubiera) antes de dividir
df = df.dropna(subset=[target])

X = df[features]
y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} — Test: {X_test.shape[0]}")

---
## 3. Preprocesamiento: numérico y categórico

Los modelos de scikit-learn solo trabajan con números. Para este dataset necesitamos:

| Columna | Tipo | Transformación necesaria |
|---|---|---|
| `pclass`, `age`, `fare` | Numérica | Imputar nulos con mediana + escalar |
| `sex`, `embark_town` | Categórica | Imputar nulos con la moda + one-hot encoding |

In [ ]:
# ─── Definir columnas por tipo ────────────────────────────────────────────────
cols_numericas    = ["pclass", "age", "fare"]
cols_categoricas  = ["sex", "embark_town"]

# ─── Pipeline para columnas numéricas ────────────────────────────────────────
# Paso 1: imputar con la mediana (robusto a outliers)
# Paso 2: escalar para que todas las features estén en la misma magnitud
preproceso_numerico = Pipeline(steps=[
    ("imputar",  SimpleImputer(strategy="median")),
    ("escalar",  StandardScaler()),
])

# ─── Pipeline para columnas categóricas ──────────────────────────────────────
# Paso 1: imputar con el valor más frecuente
# Paso 2: convertir cada categoría en una columna binaria (one-hot)
preproceso_categorico = Pipeline(steps=[
    ("imputar",  SimpleImputer(strategy="most_frequent")),
    ("codificar", OneHotEncoder(handle_unknown="ignore")),  # ignorar categorías no vistas en train
])

# ─── Combinar ambos preprocessors en un ColumnTransformer ────────────────────
preprocesador = ColumnTransformer(transformers=[
    ("num", preproceso_numerico,   cols_numericas),
    ("cat", preproceso_categorico, cols_categoricas),
])

print("Preprocesador definido.")

---
## 4. Construir el pipeline completo

In [ ]:
# ─── Pipeline completo: preprocesamiento + modelo ────────────────────────────
pipeline = Pipeline(steps=[
    ("preprocesar", preprocesador),
    ("modelo",      LogisticRegression(max_iter=1000, random_state=42)),
])

# Visualizar la estructura del pipeline
print("Estructura del pipeline:")
print(pipeline)

---
## 5. Entrenar, evaluar y reutilizar

In [ ]:
# ─── Entrenar el pipeline ─────────────────────────────────────────────────────
# Un solo .fit() ajusta el preprocesador Y entrena el modelo
pipeline.fit(X_train, y_train)

print("Pipeline entrenado.")

In [ ]:
# ─── Evaluar en test ──────────────────────────────────────────────────────────
y_pred = pipeline.predict(X_test)   # aplica preproceso y predice en un solo paso

print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print()
print(classification_report(y_test, y_pred, target_names=["No sobrevivió", "Sobrevivió"]))

In [ ]:
# ─── Usar el pipeline con datos nuevos ───────────────────────────────────────
# No hace falta preprocesar manualmente: el pipeline lo hace solo.

pasajeros_nuevos = pd.DataFrame({
    "pclass":      [1, 3],
    "sex":         ["female", "male"],
    "age":         [28.0, 22.0],
    "fare":        [75.0, 7.25],
    "embark_town": ["Cherbourg", "Southampton"],
})

predicciones = pipeline.predict(pasajeros_nuevos)
probabilidades = pipeline.predict_proba(pasajeros_nuevos)

for i, (pred, prob) in enumerate(zip(predicciones, probabilidades)):
    etiqueta = "Sobrevivió" if pred == 1 else "No sobrevivió"
    print(f"Pasajero {i+1}: {etiqueta} (confianza: {max(prob):.0%})")

---
## 6. Guardar y cargar el pipeline

Una vez que el pipeline está entrenado, podemos guardarlo en disco. Eso es útil para no tener que reentrenar cada vez, y para desplegarlo en otro entorno.

In [ ]:
# ─── Guardar el pipeline entrenado ───────────────────────────────────────────
ruta = "pipeline_titanic.joblib"
joblib.dump(pipeline, ruta)
print(f"Pipeline guardado en: {ruta}")

# ─── Cargar y usar sin reentrenar ─────────────────────────────────────────────
pipeline_cargado = joblib.load(ruta)

pred_cargado = pipeline_cargado.predict(pasajeros_nuevos)
print("Predicción desde pipeline cargado:", pred_cargado)
print("Coincide con el original:", list(pred_cargado) == list(predicciones))

---
## 7. Actividad: pipeline propio con un dataset nuevo

In [ ]:
# ─── Dataset: tips (propinas en restaurant) ───────────────────────────────────
# Tarea: predecir si la propina fue mayor al 20% del total

tips = sns.load_dataset("tips")
tips["propina_alta"] = (tips["tip"] / tips["total_bill"] > 0.20).astype(int)

print(f"Shape: {tips.shape}")
print(f"Balance: {tips['propina_alta'].value_counts().to_dict()}")
tips.head()

In [ ]:
# TODO: Construí un pipeline completo para predecir 'propina_alta'.
# Pasos sugeridos:
#   1. Elegir features (hay numéricas y categóricas)
#   2. Dividir train/test con stratify
#   3. Definir preprocesador para cada tipo de columna
#   4. Construir el pipeline con LogisticRegression o KNeighborsClassifier
#   5. Entrenar y evaluar
#   6. Predecir sobre 2 filas nuevas de ejemplo

# features_tips_num = [...]
# features_tips_cat = [...]

# ...

print("Completá las celdas TODO para continuar.")

---
## Entregable

Guardá el notebook con las celdas ejecutadas.
El entregable es el pipeline del dataset `tips` completo: preprocesador, modelo, evaluación y predicción sobre datos nuevos.

**Para la próxima clase:** vamos a comparar múltiples algoritmos de clasificación sobre el mismo dataset y elegir el mejor con criterio.